In [0]:
import pandas as pd
import numpy as np
import re
import os

import sys
sys.path.append("/Workspace/Users/karthik.raj@bio-techne.com/ScoringPhysics")

In [0]:
dbutils.widgets.text('path_design_campaign', "/Volumes/sandbox/karthik/motif_scaffolding/alpha2_beta1_double_globular_trial2")
#path_design_campaign = dbutils.widgets.get('path_design_campaign')
path_design_campaign = "/Volumes/sandbox/karthik/motif_scaffolding/alpha2_beta1_double_globular_trial3" # Should be a widget

In [0]:
path_saved_designs = f"{path_design_campaign}/rfdiffusion/designs/"
folder_designs = [x for x in os.listdir(path_saved_designs) if ("alpha" in x or "beta" in x)]

# Create Master MPNN Dataframe across all runs
list_df_mpnns = []
for folder_design in folder_designs:
  path_design_folder = path_saved_designs + folder_design + "/"
  df_design = pd.read_csv(path_design_folder + "mpnn_results.csv", index_col= 0)
  model_type = "alpha" if "alpha" in folder_design else "beta"
  df_design["rfd_model_type"] = model_type
  df_design["design_folder_path"] = path_design_folder
  df_design["design_folder_name"] = folder_design
  list_df_mpnns.append(df_design)

df_mpnn = pd.concat(list_df_mpnns).reset_index(drop=True)
df_mpnn['design_name'] = df_mpnn['design_folder_name'] + "_design" + df_mpnn['design'].astype(str) + "_n" + df_mpnn['n'].astype(str)
df_mpnn

In [0]:
df_mpnn['contig'].iloc[0]

In [0]:
import biotite
import biotite.structure as struc
from StrucTools import *

def extract_binder_contig(contigs_str: str):
    """ Extract Binder contig from contigs string to know where the motifs are for given design

        Parameters
        ----------
        contigs_str: str
            Contigs string of the overall design (includes target and binder)

        Returns
        -------
        contigs_binder: str
            Contigs string of the binder
    """
    contigs = contigs_str.split(';')
    # Store in a list initially if having multiple targets and binders
    contigs_target = []
    contigs_binder = []
    for contig in contigs:
        if "/" in contig:
            contigs_binder.append(contig)
        else:
            contigs_target.append(contig)
    # Check first if there are multiple binders
    if len(contigs_binder) > 1:
        contigs_binder = ';'.join(contigs_binder)
    else:
        contigs_binder = contigs_binder[0]
    return contigs_binder

def extract_input_motif_coords(path_input_pdb: str, contig_binder: str, desired_motif_chain_id: str = ""):
    """ Extract coordinates of the motif atoms from the input PDB providing the motif contig

        Parameters
        ----------
        path_input_pdb: str
            Path to the input PDB file
        motif_contig: str
            Contig of the motif in the input PDB file. Example: B1-21/20-20/C1-21/20-20/D1-21
        motif_chain_id: str
            Chain ID of the specific motif desired for alignment and rmsd calculation. By default, since "" will extract all motif atoms
        Returns
            all_coords_motif_input: np.array
            Coordinates of the motif atoms
    """
    # 1. Load in the pdb to get atom_array
    atom_array_input = extract_atom_array(path_input_pdb)
    all_coords_motif_input = []
    # 2. Iterate through all the contigs within the binder, may specify motif coordinates or regions to be designed
    for motif_contig in contig_binder.split('/'):
        
        # 2.1 Handle case where desired_motif_chain_id is not provided so goal is to extract all coordinates of the motif atoms from the design PDB
        if desired_motif_chain_id == "":
            motif_extraction_condition = (motif_contig[0].isalpha())
        else:
            motif_extraction_condition = (motif_contig[0] == desired_motif_chain_id)
        
        # 2.2 Now iterate through with the motif_extraction_condition
        if motif_extraction_condition:
            motif_chain_start, motif_chain_end = motif_contig.split('-')
            motif_chain_id = motif_chain_start[0]
            motif_res_start, motif_res_end = int(motif_chain_start[1:]), int(motif_chain_end)
    
            # Create input atom mask of motif atoms in the input
            atom_array_input_motif = atom_array_input[((atom_array_input.chain_id == motif_chain_id) & (atom_array_input.res_id >= motif_res_start) & (atom_array_input.res_id <= motif_res_end))]
            coords_motif_input = atom_array_input_motif.coord
            all_coords_motif_input.append(coords_motif_input)
        else:
            continue
    all_coords_motif_input = np.concatenate(all_coords_motif_input, axis = 0)
    return all_coords_motif_input

def extract_design_motif_coords(path_pdb_design: str, contig_binder: str, desired_motif_chain_id: str = "", binder_chain_id: str = "B"):
    """ Extract coordinates of the motif atoms from the design PDB providing the motif contig
        By default: Extracts all coordinates of the motif atoms from the design PDB unless a specific motif_chain_id is provided

        Parameters
        ----------
        contig_binder: str
            Contig of the motif in the input PDB file. Example: B1-21/20-20/C1-21/20-20/D1-21
        path_pdb_design: str
            Path to the design PDB file
        desired_motif_chain_id: str
            Chain ID of the specific motif desired for alignment and rmsd calculation. By default, since "" will extract all motif atoms
        binder_chain_id: str
            Chain ID of the binder in the design PDB. By default, "B" since typically "A" corresponds to target
    
    """
    # 1. Create residue-specific mask of the motif residues in the design binder sequence
    res_motif_mask = []
    for region in contig_binder.split('/'):
        
        # 2. Handle case where motif_chain_id is not provided so goal is to extract all coordinates of the motif atoms from the design PDB
        if desired_motif_chain_id == "":
            motif_extraction_condition = (region[0].isalpha())
        else:
            motif_extraction_condition = (region[0] == desired_motif_chain_id)
        
        # 3. Handle case where motif_chain_id is provided so goal is to extract coordinates of the motif atoms from the design PDB
        if motif_extraction_condition:
            #print("Region: ", region)
            preserved_region = region[1:]
            start, end = preserved_region.split('-')
            motif_len = int(end) - int(start) + 1
            motif_mask = [True] * motif_len
        else:
            design_len1, design_len2 = region.split('-')
            # Handle case of chains corresponding to those preserved from initial input
            if design_len1 != design_len2:
                design_strand_size = int(design_len2) - int(design_len1[1:]) + 1
            # Handle case of chains corresponding to those designed via RFDiffusion
            elif design_len1 == design_len2:
                design_strand_size = design_len2
            motif_mask = [False] * int(design_strand_size)
        
        res_motif_mask += motif_mask
    #print(len(res_motif_mask))
    
    atom_array_design_complex = extract_atom_array(path_pdb_design)
    seq_list, chain_starts = struc.to_sequence(atom_array_design_complex)
    seq_target, seq_binder = str(seq_list[0]), str(seq_list[1])
    #print(seq_binder)
    #print(len(seq_binder))
    atom_array_design_binder = atom_array_design_complex[atom_array_design_complex.chain_id == binder_chain_id]
    mask_atom_motif_design = struc.spread_residue_wise(atom_array_design_binder, res_motif_mask)
    atom_array_design_motif = atom_array_design_binder[mask_atom_motif_design]
    coords_motif_design = atom_array_design_motif.coord
    return coords_motif_design

In [0]:
import biotite.structure.io as bsio
def compute_chain_plddt(path_pdb_design: str, chain_id: str = 'B'):
    """ Compute pLDDT of a chain in a PDB file from Sergey's Workflow """
    struct = bsio.load_structure(path_pdb_design, extra_fields=["b_factor"])
    struct_chain = struct[struct.chain_id == chain_id]
    plddt_chain = struct_chain[struct_chain.atom_name == 'CA'].b_factor.mean()
    return plddt_chain

In [0]:
import biotite.structure as struc
from biotite.structure import superimpose, rmsd
# 1. Extract coordinates of middle strand motif
path_input_pdb = f"{path_design_campaign}/inputs/collagen_triple_helix_alpha2_model_0.pdb"
example_contigs = df_mpnn['contig'].iloc[0]
print("Example Contigs: ", example_contigs)
binder_contig = extract_binder_contig(example_contigs)
print("Binder_contigs: ", binder_contig)
all_coords_motif_input = extract_input_motif_coords(path_input_pdb, binder_contig, desired_motif_chain_id= "")
print(all_coords_motif_input.shape)

In [0]:
df_mpnn_seqs = df_mpnn['seq'].str.split('/', expand= True)
df_mpnn_seqs.rename(columns = {0 : 'seq_target', 1: 'seq_binder'}, inplace = True)
df_mpnn_seqs['len_binder'] = df_mpnn_seqs['seq_binder'].str.len()
df_mpnn.insert(loc = 8, column = 'seq_binder', value= df_mpnn_seqs['seq_binder'].values)
df_mpnn.insert(loc = 9, column = 'seq_target', value= df_mpnn_seqs['seq_target'].values)
df_mpnn.head(10)

In [0]:
import biotite.structure as struc
from biotite.structure import superimpose, rmsd

# 1. Extract coordinates of middle strand motif
# path_input_pdb = f"{path_design_campaign}/inputs/collagen_triple_helix_alpha2_model_0.pdb"
example_contigs = df_mpnn['contig'].iloc[0]
print("Example Contigs: ", example_contigs)
binder_contig = extract_binder_contig(example_contigs)
print("Binder_contigs: ", binder_contig)
desired_motif_chain_id = "C"
all_coords_motif_input = extract_input_motif_coords(path_input_pdb, binder_contig, desired_motif_chain_id= "")
desired_coords_motif_input = extract_input_motif_coords(path_input_pdb, binder_contig, desired_motif_chain_id= desired_motif_chain_id)
print("Desired Coordinates Motif Shape: ", desired_coords_motif_input.shape)
print("All Coordinates Motif Shape: ", all_coords_motif_input.shape)

list_rmsd_middle_strand = []
list_rmsd_motif = []
list_design_paths = []
list_plddt_binder = []
for index, row in df_mpnn.iterrows():
    
    # 2. Extract path of the pdb design
    path_pdb_folder = row['design_folder_path'] + 'all_pdb' + "/"
    path_pdb_design = path_pdb_folder + f"design{row['design']}_n{row['n']}.pdb"

    # 2.5: Compute Binder PLDDT:
    plddt_binder = compute_chain_plddt(path_pdb_design, chain_id= "B")
    
    # 3. Extract contigs as a string
    contigs_str = str(row['contig'])
    print("Contigs_str: ", contigs_str)
    binder_contig = extract_binder_contig(contigs_str)
    print("Binder_Contig: ", binder_contig)
    
    # 4. Extract coordinates of full motif and motif middle strand in the design
    all_coords_motif_design = extract_design_motif_coords(path_pdb_design, binder_contig, desired_motif_chain_id= "")
    desired_coords_motif_design = extract_design_motif_coords(path_pdb_design, binder_contig, desired_motif_chain_id=  desired_motif_chain_id)
    print("All Coordinates Motif Design Shape: ", all_coords_motif_design.shape)
    print("Desired Coordinates Motif Design Shape: ", desired_coords_motif_design.shape)
    
    # 5. Align all input and design motif coordinates
    all_aligned_design_coords, applied_transformation = superimpose(all_coords_motif_input, all_coords_motif_design)
    # 5. Calculate RMSD between  all motif input and all motif design
    rmsd_motif = rmsd(all_coords_motif_input, all_aligned_design_coords)

    # 6. Align desired input and design motif coordinates
    desired_aligned_design_coords, applied_transformation = superimpose(desired_coords_motif_input, desired_coords_motif_design)
    # 6. Calculate RMSD between desired input and desired motif design
    rmsd_middle_strand = rmsd(desired_coords_motif_input, desired_aligned_design_coords)
    
    # 7. Append RMSD & Binder-Specific PLDDT & Design PDB Path to respective lists
    list_rmsd_motif.append(rmsd_motif)
    list_rmsd_middle_strand.append(rmsd_middle_strand)
    list_plddt_binder.append(plddt_binder)
    list_design_paths.append(path_pdb_design)

df_mpnn.insert(loc = 3, column = "plddt_binder", value = list_plddt_binder)
df_mpnn.insert(loc = 9, column = "rmsd_middle_strand", value = list_rmsd_middle_strand)
df_mpnn.insert(loc = 10, column = "rmsd_motif", value = list_rmsd_motif)
df_mpnn['design_pdb_path'] = list_design_paths
df_mpnn.head()



In [0]:
df_mpnn['rmsd_middle_strand'].describe()
df_mpnn['rmsd_middle_strand'].hist(bins= 50)

In [0]:
import seaborn as sns
import matplotlib.pyplot as plt

def plot_histogram_seaborn(df, column, bins=50, figsize=(10, 5), **kwargs):
    fig, ax = plt.subplots(figsize=figsize)
    sns.histplot(df[column], bins=bins, ax=ax, **kwargs)
    ax.set_title(f"{column} distribution")
    ax.set_xlabel(column)
    ax.set_ylabel("Frequency")

column = 'plddt_binder'
plot_histogram_seaborn(df_mpnn, column, bins=50)

In [0]:
# Create Kernel Density Plot of MPNN RMSD Motif
import seaborn as sns
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(df_mpnn["rmsd_motif"], fill=True, ax= ax, label = 'rmsd_motif', alpha = 0.5)
sns.histplot(df_mpnn['rmsd_middle_strand'], fill = True, ax = ax, label = 'rmsd_middle_strand', alpha = 0.5)
plt.title("Triple Helix Motif RMSD from MPNN Produced Sequences")
plt.xlabel("RMSD Motif")
plt.ylabel("Density")
plt.legend(loc = 'upper right')
plt.show()

In [0]:
agg_funcs = ['mean', 'std', 'count']
df_mpnn_copy = df_mpnn.copy()
df_mpnn_groupedby = df_mpnn_copy.groupby(['rfd_model_type', 'contig']).agg({'rmsd_middle_strand' : agg_funcs,
                                                                             'rmsd_motif': agg_funcs,
                                                                             'rmsd' : agg_funcs,
                                                                             'plddt' : agg_funcs,
                                                                             'plddt_binder' : agg_funcs,
                                                                             'ptm' : agg_funcs,
                                                                             'pae' : agg_funcs
                                                                              }).sort_values([('rmsd_middle_strand', 'mean')], ascending= True)
df_mpnn_groupedby.reset_index(inplace = True)
df_mpnn_groupedby

In [0]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.scatterplot(x = df_mpnn['rmsd_motif'], y = df_mpnn['rmsd_middle_strand'], ax = ax, label = 'triple helix vs middle strand')

plt.plot([df_mpnn['rmsd_motif'].min(), df_mpnn['rmsd_motif'].max()],
         [df_mpnn['rmsd_motif'].min(), df_mpnn['rmsd_motif'].max()],
         color='red', linestyle='-', linewidth=2, label='y = x')

plt.title("RMSD Motif vs RMSD Middle Strand")
plt.xlabel("RMSD Motif")
plt.ylabel("RMSD Middle Strand")
plt.legend()
plt.show()

### Visualizing RFDiffusion Designed Structures

In [0]:
mean_rmsd_middle_strand = df_mpnn['rmsd_middle_strand'].describe()['50%']
mean_rmsd_motif = df_mpnn['rmsd_motif'].describe()['50%']
print(f"Mean RMSD Middle Strand: {mean_rmsd_middle_strand}")
print(f"Mean RMSD Motif: {mean_rmsd_motif}")

In [0]:
threshold_plddt_binder = 65
threshold_rmsd_middle_strand = mean_rmsd_middle_strand
threshold_rmsd_motif = mean_rmsd_motif
threshold_rmsd_holo_binder = 2
df_filtered = df_mpnn[(df_mpnn['rmsd_middle_strand'] < threshold_rmsd_middle_strand) & 
                      (df_mpnn['rmsd_motif'] <  threshold_rmsd_motif) & 
                      (df_mpnn['rmsd_holo_binder_rfd_af2'] < threshold_rmsd_holo_binder) &
                      (df_mpnn['plddt_binder'] > threshold_plddt_binder)]
print("Number of Passing Designs for Boltz2 Evaluation: ", len(df_filtered))
df_filtered

In [0]:
import py2Dmol
from StrucTools import visualize_structure
import random
df_data = df_filtered.copy()
random_index = random.randint(0, len(df_data))
#random_index = 100  #(bad design in globular double strand is with index of 4097)
design_path = df_data['design_pdb_path'].iloc[random_index]
print(f"Random index: {random_index}")
print(f"Design Name: {df_data['design_name'].iloc[random_index]}")
print(f"Design Path: {design_path}")
print(f"RMSD Middle Strand: {df_data['rmsd_middle_strand'].iloc[random_index]}")
print(f"RMSD: {df_data['rmsd'].iloc[random_index]}")
print(f"RMSD Motif: {df_data['rmsd_motif'].iloc[random_index]}")
print(f"RMSD Holo Binder RFD AF2: {df_data['rmsd_holo_binder_rfd_af2'].iloc[random_index]}")
print(f"RFD Model Type: {df_data['rfd_model_type'].iloc[random_index]}")
print(f"PLDDT: {df_data['plddt'].iloc[random_index]}")
print(f"PLDDT Binder: {df_data['plddt_binder'].iloc[random_index]}")
print(f"PTM: {df_data['ptm'].iloc[random_index]}")
print(f"PAE: {df_data['pae'].iloc[random_index]}")
print(f"Contigs: {df_data['contig'].iloc[random_index]}")
visualize_structure(design_path)

### Save into Final DataFrame & Split into 4 DataFrames for Parallel Execution

In [0]:
df_filtered_final = df_filtered.reset_index(drop = True)
df_filtered_final.head()

In [0]:
path_design_campaign

In [0]:
csv_save_path = f"{path_design_campaign}/rfdiffusion/"
df_filtered_final.to_csv(csv_save_path + "rfd_af2_passed_designs.csv")

In [0]:
rfd_af2_passed_chunks_save_folder = f"{path_design_campaign}/rfdiffusion/rfd_af2_passed_boltz_input/"
for index, df_chunk in enumerate(np.array_split(df_filtered_final, 3)):
    df_chunk.to_csv(f"{rfd_af2_passed_chunks_save_folder}/rfd_af2_passed_chunk_{index}.csv")